In [40]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder\
        .master("local[*]")\
        .appName("test")\
        .getOrCreate()

In [41]:
df_green = spark.read.option("recursiveFileLookup","true").parquet("data/pq/green")

In [42]:
df_green.createOrReplaceTempView('green')

In [43]:
df_green_revenue = spark.sql("""
SELECT 
    -- Reveneue grouping 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    
    SUM(tip_amount) AS AMOUNT,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [44]:
df_green_revenue\
    .repartition(20)\
    .write.parquet('data/report/revenue/green',mode='overwrite')

In [45]:
df_yellow = spark.read.option("recursiveFileLookup","true").parquet("data/pq/yellow")

In [46]:
df_yellow.createOrReplaceTempView('yellow')

In [47]:
df_yellow_revenue = spark.sql("""
SELECT 
    -- Reveneue grouping 
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    
    SUM(tip_amount) AS AMOUNT,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [48]:
df_yellow_revenue\
    .repartition(20)\
    .write.parquet('data/report/revenue/yellow',mode='overwrite')

In [51]:
df_green_revenue_tmp = df_green_revenue\
        .withColumnRenamed('AMOUNT','green_amount')\
        .withColumnRenamed('number_records','green_number_records')

In [66]:
df_yellow_revenue_tmp = df_yellow_revenue\
        .withColumnRenamed('AMOUNT','yellow_amount')\
        .withColumnRenamed('number_records','yellow_number_records')

## Spark JOIN

In [67]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp,on=['hour','zone'],how='outer')

In [68]:
df_join.write.parquet('data/report/revenue/total')

### Joining Bigger Dataset With Smaller Dataset

In [73]:
df_revenue = spark.read.parquet('data/report/revenue/total')

In [75]:
df_zones = spark.read.parquet('zones')

In [82]:
df_revenue_zone = df_revenue.join(df_zones,df_revenue.zone == df_zones.LocationID)

In [90]:
df_revenue_zone.drop('LocationID','zone').write.parquet('tmp/revenue-zone',mode='overwrite')

In [83]:
df_revenue_zone.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|  Borough|                Zone|service_zone|
+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2020-01-01 00:00:00|   4|              NULL|                NULL|113.76000000000002|                   57|         4|Manhattan|       Alphabet City| Yellow Zone|
|2020-01-01 00:00:00|  10|              NULL|                NULL|               0.0|                    2|        10|   Queens|        Baisley Park|   Boro Zone|
|2020-01-01 00:00:00|  12|              NULL|                NULL|11.700000000000001|                    6|        12|Manhattan|        Battery Park| Yellow Zone|
|2020-01-01 00:00:00| 

In [76]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly